# Evaluate Term Dispersion Scores on the GENIA Corpus Data and Reproduce Results Reported in the Mannuscript 

Authors: Samuel Sarria Hurtado, Uyen "Rachel" Lai, and Paul Sheridan

Description: Evaluate the following term dispersion scores on the Genia corpus data:
- Inverse Document Frequency (IDF)
- Inverse Collection Frequency (ICF)
- Chi-square
- Church and Gale (CG)
- Irvine and Callison-Burch (ICB)
- Derivation of Proportions (DoP)
- Residual ICF (RICF)

Calculate average P@k scores for each scoring function using the GENIA terms as ground truth. Also, evaluate scoring functions for their ability to filter out stopwords. 

## Preliminaries

In [1]:
# Imports
import sys
import os
import pickle
import json
import pandas as pd
sys.path.append('../../')
import wordstats
from sklearn.feature_extraction.text import CountVectorizer
import random
import numpy as np
import scipy
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from io import StringIO
from numpy import nan
from tqdm import tqdm
import rbo

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pasheridan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load the GENIA Corpus Data

In particular, we load the preprocessed GENIA corpus documents, and gold standard biological terms (i.e., lexical units) and their associated semantic classes (i.e., sems) and associated high-level class (i.e., amino_acid, nucleotide, multi_cell, cell, and other).

First, load the corpus docs, and the lexical units. Then hardcode the high-level semantic classes.

In [2]:
# Load the preprocessed GENIA corpus documents
genia_corpus_path = '../1-preprocessing/GENIAcorpus3.02-preprocessed.json'

with open(genia_corpus_path, "r") as j:
  genia_corpus = json.loads(j.read())

# Load gold standard terms 
genia_keywords_path = '../1-preprocessing/GENIAcorpus3.02-keywords.tsv'

with open(genia_keywords_path, "r") as c:
  genia_lexical_units_and_sems = pd.read_csv(c, sep='\t')

genia_lexical_units = genia_lexical_units_and_sems.lex.to_numpy()

# Hardcode the low-level semantic classes and their associated high-level abstract semantic classes
amino_acid_sems = ['G#amino_acid_monomer', 'G#peptide', 'G#protein_N/A',
              'G#protein_complex', 'G#protein_domain_or_region',
              'G#protein_family_or_group', 'G#protein_molecule',
              'G#protein_substructure', 'G#protein_subunit',
              'G#other_organic_compound', 'G#organic', 'G#inorganic', 'G#atom',
              'G#carbohydrate', 'G#lipid']
nucleotide_sems = ['G#nucleotide', 'G#polynucleotide', 'G#DNA_N/A',
        'G#DNA_domain_or_region', 'G#DNA_family_or_group', 'G#DNA_molecule',
        'G#DNA_substructure', 'G#RNA_N/A', 'G#RNA_domain_or_region',
        'G#RNA_family_or_group', 'G#RNA_molecule', 'G#RNA_substructure']
multi_cell_sems = ['G#virus', 'G#mono_cell', 'G#multi_cell', 'G#body_part', 'G#tissue']
cell_sems = ['G#cell_type', 'G#cell_component', 'G#cell_line', 'G#other_artificial_source']
other_sems = ['G#other_name']
high_level_semantic_class_names = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']
high_level_semantic_class_lex_units = [genia_lexical_units, amino_acid_sems, nucleotide_sems, multi_cell_sems, cell_sems, other_sems]

Process high-level semantic classes.

In [3]:
# Collect lexical units belonging to a given high-level semantic class
def get_high_level_semantic_class_words(high_level_class_lst):
  words = []
  for k, v in lex_sem_dct.items():
    if v in high_level_class_lst:
      words.append(k)
  return words

# Create dictionary of lexical units and their associated semantic classes
sem = np.array(genia_lexical_units_and_sems['sem'])
lex = np.array(genia_lexical_units_and_sems['lex'])
lex_sem_dct = dict(zip(lex, sem))

# Create data frame of lexical units, semantic classes, and high-level semantic classes
lex_size = len(genia_lexical_units_and_sems) # Number of lexical units in the vocabulary
high_level_sems_lst = [] # Initialize list for recording high-level semantic classes

# For each term in the vocab, identify low-level semantic class with high-level one 
for index in range(lex_size):
    low_level_sem = genia_lexical_units_and_sems.iloc[index, 1]
    if low_level_sem in amino_acid_sems:
        high_level_sems_lst.append('amino_acid')
    elif low_level_sem in nucleotide_sems:
        high_level_sems_lst.append('nucleotide')
    elif low_level_sem in multi_cell_sems:
        high_level_sems_lst.append('multi_cell')
    elif low_level_sem in cell_sems:
        high_level_sems_lst.append('cell')
    else:
        high_level_sems_lst.append('other')

# Add high-level semantic classes to data frame
genia_lexical_units_and_sems['class'] = high_level_sems_lst

# Print to console:
display(genia_lexical_units_and_sems)

,lex,sem,class
0,IL-2_gene_expression_lex,G#other_name,other
1,IL-2_gene_lex,G#DNA_domain_or_region,nucleotide
2,NF-kappa_B_activation_lex,G#other_name,other
3,NF-kappa_B_lex,G#protein_molecule,amino_acid
4,CD28_lex,G#protein_molecule,amino_acid
...,...,...,...
31782,gp160-induced_AP-1_complex_lex,G#protein_complex,amino_acid
31783,protein_synthesis-independent_lex,G#other_name,other
31784,calcium_channel_blocker_lex,G#other_organic_compound,amino_acid
31785,anti-CD3-induced_interleukin-2_secretion_lex,G#other_name,other


## Prepare the GENIA Corpus Data for Analysis

Prepare the corpus vocabulary.

In [4]:
# Compile the GENIA corpus vocabulary
pre_vocab = []
for i in range(len(genia_corpus)):
  pre_vocab.append(genia_corpus[i].split())

vocab = []
for i in range(len(pre_vocab)):
  for j in range(len(pre_vocab[i])):
    vocab.append(pre_vocab[i][j])

vocab = list(set(vocab))
vocab.sort()

# Helper function to ensure that CountVectorizer doesn't ignore any terms
def analyzer_custom(doc):
  return doc.split()

# Convert GENIA documents into term-in-document matrix of token counts.
counter = CountVectorizer(lowercase=False, vocabulary=vocab, analyzer=analyzer_custom)
collection = counter.transform(genia_corpus)

## Evaluate Term Dispersion Scores for Selected Measures

Calculate bag-of-words model word statistics and related quantities.

In [5]:
# Calculate word statistics and related quantities
m = len(counter.get_feature_names_out()) # vocab size
d = collection.shape[0] # collection size
N_i = wordstats.get_Ni(collection)
N_j = wordstats.get_Nj(collection)
N = wordstats.get_N(N_j)
B_ij = wordstats.get_Bij(collection)
B_i = wordstats.get_Bi(B_ij)
B_j = wordstats.get_Bj(B_ij)
DF = wordstats.get_DF(B_i, d)
CF = wordstats.get_CF(N_i)
nij_by_nj = wordstats.get_nij_by_nj(collection, N_j)
thetas = np.array(range(1, max(N_i.A[0]) + 1))/N
opt_thetas = wordstats.get_opt_thetas(N, m, d, N_i, N_j, B_i, thetas)

Evaluate term dispersion scores.

In [6]:
# Calculate word dispersion scores according the various measures used in this study
IDF = wordstats.get_IDF(DF)
ICF = wordstats.get_ICF(CF)
Chisq = wordstats.get_Chisq(collection)
CG = wordstats.get_CG(N_i, B_i)
ICB = wordstats.get_ICB(nij_by_nj, B_i)
DoP = wordstats.get_DoP(collection, N_i, N_j, N)
RICF = wordstats.get_RICF(opt_thetas, N, ICF)

/Users/pasheridan/Desktop/github-repos/bursty-term-measure/genia/3-tables/../../wordstats.py:209: RuntimeWarning: divide by zero encountered in log
  return -np.log(chisq_values)


Arrange term dispersion scores into a data frame.

In [7]:
# Initialize term dispersion scores data frame (augmented with ni and bi values)
term_scores_aug_df = pd.DataFrame(data=
                    {'lex': counter.get_feature_names_out(),
                     'IDF': IDF.A[0],
                     'ICF': ICF.A[0],
                     'Chi-sq': Chisq,
                     'CG': CG.A[0],
                     'ICB': ICB.A[0],
                     'DoP': DoP.A[0],
                     'RICF': RICF.A[0],
                     'bi': B_i.A[0],
                     'ni': N_i.A[0]})

# Augment with low-level and high-level semantic classes
term_scores_aug_df = pd.merge(term_scores_aug_df, genia_lexical_units_and_sems, on='lex', how='left')

# Tidy up the data frame
new_order = ['lex', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'RICF'] # Define column ordering
term_scores_aug_df = term_scores_aug_df.reindex(columns=new_order)
term_scores_aug_df = term_scores_aug_df.rename(columns={'lex': 'term'}) # Rename 'lex' column to 'term'
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep=False)] # There are a few duplicate rows for some unknown reason
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

# Print to console
print("Term dispersion scores:")
display(term_scores_aug_df)

# Write to TSV
term_scores_aug_df.to_csv('term-dispersion-scores.tsv', sep='\t')

Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
16653,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.404383
16654,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.404383
29777,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.692896
29778,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.692896
32036,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.808753
32037,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.808753


Term dispersion scores:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...
40802,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40803,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40804,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,-0.000251
40805,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,-0.000251


## Compile GENIA Corpus Summary Statistics

This is the result of Table 3 from the manuscript.

In [38]:
# Desginated ordering for the high-level semantic classes
high_level_semantic_class_ord = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']

# Count number of semantic subclasses in each high-level class
subclass_counts = [len(amino_acid_sems), len(nucleotide_sems), len(multi_cell_sems), len(cell_sems), len(other_sems)]

# Count number of distinct lexical units in each high-level semantic class
lex_unit_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['term'].nunique().reindex(high_level_semantic_class_ord).to_list()

# Count number of annotations associated with each high-level semantic class
annotation_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Count number of singletons associated with each high-level semantic class
singleton_counts = term_scores_aug_df[term_scores_aug_df['ni'] == 1].dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Initialize GENIA summary statistics data frame
genia_summary_stats_df = pd.DataFrame({
    'Semantic class': high_level_semantic_class_ord,
    'Sub-class': subclass_counts,
    'Unique terms': lex_unit_counts,
    'Annotations': annotation_counts,
    'Singletons': singleton_counts
})

# Print GENIA summary statistics to console
display(genia_summary_stats_df)

# Write to CSV
os.makedirs('table-3', exist_ok=True)
genia_summary_stats_df.to_csv('table-3/semantic-class-stats.csv', index=False)

,Semantic class,Sub-class,Unique terms,Annotations,Singletons
0,amino_acid,15,10155,42478,6571
1,nucleotide,12,5574,11619,4115
2,multi_cell,5,1444,5247,961
3,cell,4,4051,11626,2956
4,other,1,10560,19999,8071


## Terminology Extraction Task Experiment

Here we reproduce the result of Tables 5, 6, 7, 12, 13, and 14 from the manuscript.

In [40]:
# Create a minimal data frame of term dispersion scores
term_scores_df = term_scores_aug_df[term_scores_aug_df['ni'] > 1] # Filter out singletons
term_scores_df = term_scores_df.reset_index(drop=True) # Reinitialize row indices
term_scores_df = term_scores_df.drop(columns=['sem', 'class', 'ni', 'bi'])

# Print to console
display(term_scores_df)

,term,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,(+)-pentazocine_lex,7.600902,12.107806,311.074036,2.0,522.0,-0.000720,0.692896
1,-120_lex,7.600902,12.107806,311.074036,2.0,360.0,-0.000496,0.692896
2,-150_bp_lex,7.600902,11.702341,inf,3.0,711.0,-0.000654,1.098361
3,-201/-184_NXS_lex,7.600902,12.107806,311.074036,2.0,482.0,-0.000665,0.692896
4,-201_and_-130_lex,7.600902,12.107806,311.074036,2.0,482.0,-0.000665,0.692896
...,...,...,...,...,...,...,...,...
14149,zinc_finger_region_lex,6.907755,12.107806,0.688948,1.0,200.5,-0.001106,-0.000532
14150,zinc_finger_transcription_factor_lex,5.991465,10.855043,122.211280,1.4,201.4,-0.002168,0.335116
14151,zinc_lex,6.907755,10.721512,inf,4.0,447.0,-0.000576,1.385763
14152,zone,6.907755,12.107806,0.688948,1.0,334.5,-0.001845,-0.000532


Define various functions used in the analysis.

In [41]:
# Grab the top k terms
def top_k(dct, k):
  keys = dct.keys()
  values = []
  for key in keys:
    values.append(dct[key][:k])
  keys_values_pair = zip(keys, values)
  return dict(keys_values_pair)

# Count up terms
def count_words(lst, imp_words):
  counter = 0
  for x in lst:
    if x in imp_words:
      counter += 1
  return counter

# Randomly resort term dispersion scores data frame
def resort(term_scores_df):
  sorted_terms = []
  bursty_measure_names = term_scores_df.columns.values.tolist()[1:]

  for measure in bursty_measure_names:
      # Copy the data frame and add a random column
      temp_df = term_scores_df.copy()
      temp_df['random'] = np.random.rand(len(temp_df))
        
      # Sort by the measure and the random column
      sorted_df = temp_df[['term', measure, 'random']].sort_values(by=[measure, 'random'], ascending=[False, True])
        
      # Append the sorted terms to the list
      sorted_terms.append(np.array(sorted_df['term']))
        
      # Drop the random column from the temporary data frame
      temp_df.drop(columns='random', inplace=True)
    
  sorted_terms = np.array(sorted_terms)
  measure_term_pair = zip(bursty_measure_names, sorted_terms)
  sorted_measures = dict(measure_term_pair)
    
  return sorted_measures
    
# Calculate Precision at k scores
def calc_pk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  pk_dct = dict(zip(measures, counts))  
  for measure in pk_dct.keys():
      for k in k_values:
          pk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/k)
  result = pd.DataFrame(pk_dct, index=k_values)
  return result

# Calculate Recall at k scores
def calc_rk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rk_dct = dict(zip(measures, counts))  
  for measure in rk_dct.keys():
      for k in k_values:
          rk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/len(lex_units))
  result = pd.DataFrame(rk_dct, index=k_values)
  return result

# Calculate F1 at k scores
def calc_fk(pk, rk):
  result = 2 * (pk * rk) / (pk + rk)
  result = result.fillna(0) # Nan scores are redefined as 0
  return result

# Calculate Rank Biased Overlap scores
def calc_rbo(sorted_measures, k_values):
  RICF = sorted_measures["RICF"]
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))  
  for measure in rbo_dct.keys():
      for k in k_values:
          S = top_k(sorted_measures, k)[measure]
          T = RICF[0:k]
          rbo_dct[measure].append(rbo.RankingSimilarity(S, T).rbo())
  result = pd.DataFrame(rbo_dct, index=k_values)
  return result

# Calculate Rank Biased Overlap scores for each semantic class
def calc_rbo2(sorted_measures, categories):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))
  #rbo_dct = dict.fromkeys(measures, [])

  for measure in measures:
      l1 = sorted_measures[measure].tolist()
      
      for category, lex_units in categories.items():
          S = sorted(set(l1) & set(lex_units), key = l1.index)
          l2 = sorted_measures["RICF"].tolist()
          RICF = sorted(set(l2) & set(lex_units), key = l2.index)          
          rbo_dct[measure].append(rbo.RankingSimilarity(S, RICF).rbo()) 

  result = pd.DataFrame(rbo_dct, index=categories.keys())
  return result
    
# Calculate mean P@k, R@k, and F1@k scores
def calc_score_means(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        means = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            mean_values = np.mean(values, axis=0)
            means[column] = mean_values
        result.append(pd.DataFrame(means, index=nested_list[0][i].index))
    return result

# Calculate standard deviations of P@k, R@k, and F1@k scores
def calc_score_sds(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        std_devs = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            std_values = np.std(values, axis=0, ddof=1)
            std_devs[column] = std_values
        result.append(pd.DataFrame(std_devs, index=nested_list[0][i].index))
    return result

# Calculate mean RBO scores
def calc_mean_rbo_scores(scores_list):
  R = len(scores_list) # Number of replicates
  H = len(scores_list[0]) # Number of dispersion metrics
  d_metrics = scores_list[0].columns # Dispersion metrics by name
  k_values = scores_list[0].index # Top k values
  result = {}
    
  for d_metric in d_metrics:
    scores = [scores_list[r][d_metric].values for r in range(R)]
    mean_scores = np.mean(scores, axis=0)
    result[d_metric] = mean_scores
        
  return pd.DataFrame(result, index=k_values)

Evaluate Precision at k, Recall at k, F1 at k, and RBO scores.

In [42]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Set number of replicates
R = 3 # For testing purposes; change back to 100 for final results

# These are the Precision @ k, Recall @ k and Rank Biased Overlap scores
all_pk_scores = []
all_rk_scores = []
all_fk_scores = []
all_rbo_scores = []
all_rbo_scores2 = []

# Used as inputs for calculating the various scores
k_values = np.array([10, 50, 100, 500, 1000, 5000])
categories = {
    'all': genia_lexical_units,
    'amino_acid': amino_acid,
    'nucleotide': nucleotide,
    'multi_cell': multi_cell,
    'cell': cell,
    'other': other}

# Calculate evaluation metrics
for r in tqdm(range(R)):
    print('r =', r)
    pk_scores = []
    rk_scores = []
    fk_scores = []
    sorted_measures = resort(term_scores_df)
    
    for category, lex_units in categories.items():
        pk = calc_pk(lex_units, sorted_measures, k_values)
        pk_scores.append(pk)
        rk = calc_rk(lex_units, sorted_measures, k_values)
        rk_scores.append(rk)
        fk = calc_fk(pk, rk)
        fk_scores.append(fk)
    
    all_pk_scores.append(pk_scores)
    all_rk_scores.append(rk_scores)
    all_fk_scores.append(fk_scores)
    rbo_scores = calc_rbo(sorted_measures, k_values)
    all_rbo_scores.append(rbo_scores)
    rbo_scores2 = calc_rbo2(sorted_measures, categories)
    all_rbo_scores2.append(rbo_scores2)

  0%|                                                                                                | 0/3 [00:00<?, ?it/s]

r = 0


 33%|█████████████████████████████▎                                                          | 1/3 [00:44<01:28, 44.22s/it]

r = 1


 67%|██████████████████████████████████████████████████████████▋                             | 2/3 [01:29<00:44, 44.64s/it]

r = 2


100%|████████████████████████████████████████████████████████████████████████████████████████| 3/3 [02:14<00:00, 44.73s/it]


Save evaluation metrics as Pkl files.

In [43]:
# Ensure the directory exists
os.makedirs('scores-dump', exist_ok=True)

# Write P@k scores to Pkl
with open('scores-dump/all_pk_scores.pkl', 'wb') as file:
    pickle.dump(all_pk_scores, file)

# Write R@k scores to Pkl
with open('scores-dump/all_rk_scores.pkl', 'wb') as file:
    pickle.dump(all_rk_scores, file)

# Write F1@k scores to Pkl
with open('scores-dump/all_fk_scores.pkl', 'wb') as file:
    pickle.dump(all_fk_scores, file)

# Write RBO scores to Pkl
with open('scores-dump/all_rbo_scores.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores, file)

# Write RBO scores as calculated for each semantic class to Pkl
with open('scores-dump/all_rbo_scores2.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores2, file)

In [44]:
# Calculate mean P@k scores and write to CSV
all_pk_scores_means = calc_score_means(all_pk_scores)
os.makedirs('table-5', exist_ok=True)
pd.DataFrame(all_pk_scores_means[0]).to_csv('table-5/all-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[1]).to_csv('table-5/amino_acid-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[2]).to_csv('table-5/nucleotide-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[3]).to_csv('table-5/multicell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[4]).to_csv('table-5/cell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[5]).to_csv('table-5/other-pk-means.csv', index=False)

# Calculate standard deviations for P@k scores and write to CSV
all_pk_scores_sds = calc_score_sds(all_pk_scores)
os.makedirs('table-12', exist_ok=True)
pd.DataFrame(all_pk_scores_sds[0]).to_csv('table-12/all-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[1]).to_csv('table-12/amino_acid-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[2]).to_csv('table-12/nucleotide-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[3]).to_csv('table-12/multicell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[4]).to_csv('table-12/cell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[5]).to_csv('table-12/other-pk-sds.csv', index=False)

In [45]:
# Display mean P@k scores and console
print("Mean P@k scores:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_means[0].round(4))
    display(all_pk_scores_means[1].round(4))
    display(all_pk_scores_means[2].round(4))
    display(all_pk_scores_means[3].round(4))
    display(all_pk_scores_means[4].round(4))
    display(all_pk_scores_means[5].round(4))

Mean P@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.8667,0.7000,0.9333,1.0000,1.0000,1.0000,1.0000
50,0.9133,0.7200,0.9333,0.9600,1.0000,0.9600,1.0000
100,0.9200,0.7167,0.9367,0.9800,0.9800,0.9400,1.0000
500,0.9213,0.7493,0.9513,0.9840,0.9740,0.9560,0.9920
1000,0.9287,0.7677,0.9520,0.9807,0.9643,0.9517,0.9850
5000,0.8799,0.7681,0.9146,0.9275,0.8992,0.8882,0.9321


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.4333,0.2000,0.5000,1.0000,0.8000,0.2000,1.0000
50,0.4067,0.2000,0.5533,0.7400,0.7000,0.4000,0.7933
100,0.3867,0.2133,0.5400,0.8200,0.7467,0.4600,0.8300
500,0.3700,0.2387,0.5407,0.6907,0.6220,0.4340,0.6940
1000,0.3820,0.2393,0.5457,0.6457,0.5900,0.4180,0.6420
5000,0.3549,0.2515,0.4301,0.4284,0.4144,0.3614,0.4299


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.2000,0.2333,0.1667,0.0000,0.1000,0.2000,0.0000
50,0.1600,0.1333,0.1333,0.1000,0.1200,0.1200,0.1000
100,0.1933,0.1267,0.1567,0.1000,0.1000,0.1300,0.1100
500,0.1667,0.1333,0.1600,0.1413,0.1460,0.1340,0.1433
1000,0.1743,0.1403,0.1560,0.1320,0.1400,0.1580,0.1367
5000,0.1564,0.1329,0.1521,0.1553,0.1514,0.1552,0.1557


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0000,0.0000,0.0333,0.0000,0.0000,0.0000,0.0000
50,0.0400,0.0667,0.0467,0.0400,0.0400,0.0200,0.0400
100,0.0400,0.0467,0.0367,0.0200,0.0200,0.0200,0.0200
500,0.0487,0.0427,0.0387,0.0327,0.0320,0.0660,0.0333
1000,0.0467,0.0427,0.0360,0.0350,0.0413,0.0580,0.0367
5000,0.0437,0.0411,0.0439,0.0433,0.0410,0.0454,0.0440


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0333,0.1667,0.0000,0.0000,0.0000,0.1000,0.0000
50,0.0800,0.1333,0.0667,0.0200,0.0000,0.0800,0.0067
100,0.0833,0.1100,0.0733,0.0100,0.0200,0.0600,0.0100
500,0.1033,0.0973,0.0693,0.0400,0.0600,0.0880,0.0407
1000,0.1057,0.0983,0.0747,0.0610,0.0697,0.0910,0.0607
5000,0.1013,0.0948,0.0949,0.1006,0.1011,0.0994,0.1007


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.2000,0.1000,0.2333,0.0000,0.1000,0.5000,0.0000
50,0.2267,0.1867,0.1333,0.0600,0.1400,0.3400,0.0600
100,0.2167,0.2200,0.1300,0.0300,0.0933,0.2700,0.0300
500,0.2327,0.2373,0.1427,0.0793,0.1140,0.2340,0.0807
1000,0.2200,0.2470,0.1397,0.1070,0.1233,0.2267,0.1090
5000,0.2235,0.2478,0.1936,0.1999,0.1913,0.2268,0.2019


In [46]:
# Displaye standard deviation of P@k scores to console
print("P@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_sds[0].round(4))
    display(all_pk_scores_sds[1].round(4))
    display(all_pk_scores_sds[2].round(4))
    display(all_pk_scores_sds[3].round(4))
    display(all_pk_scores_sds[4].round(4))
    display(all_pk_scores_sds[5].round(4))

P@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.1528,0.0000,0.0577,0.0000,0.0000,0.0000,0.0000
50,0.0231,0.0800,0.0416,0.0000,0.0000,0.0000,0.0000
100,0.0000,0.0513,0.0231,0.0000,0.0000,0.0000,0.0000
500,0.0042,0.0155,0.0050,0.0000,0.0000,0.0000,0.0000
1000,0.0023,0.0112,0.0053,0.0015,0.0006,0.0006,0.0010
5000,0.0017,0.0002,0.0008,0.0006,0.0000,0.0000,0.0006


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.2309,0.2000,0.2646,0.0000,0.0000,0.0,0.0000
50,0.0611,0.0600,0.1361,0.0000,0.0000,0.0,0.0115
100,0.0153,0.0252,0.0781,0.0000,0.0058,0.0,0.0000
500,0.0183,0.0083,0.0186,0.0042,0.0000,0.0,0.0020
1000,0.0020,0.0029,0.0153,0.0029,0.0000,0.0,0.0020
5000,0.0022,0.0001,0.0006,0.0013,0.0000,0.0,0.0005


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0000,0.1528,0.0577,0.0000,0.0,0.0,0.0000
50,0.0346,0.0902,0.0611,0.0000,0.0,0.0,0.0000
100,0.0252,0.0462,0.0473,0.0000,0.0,0.0,0.0000
500,0.0110,0.0117,0.0209,0.0031,0.0,0.0,0.0031
1000,0.0055,0.0032,0.0147,0.0030,0.0,0.0,0.0059
5000,0.0017,0.0001,0.0009,0.0009,0.0,0.0,0.0001


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0000,0.0000,0.0577,0.0000,0.0000,0.0,0.0000
50,0.0400,0.0115,0.0231,0.0000,0.0000,0.0,0.0000
100,0.0173,0.0115,0.0231,0.0000,0.0000,0.0,0.0000
500,0.0076,0.0023,0.0081,0.0012,0.0000,0.0,0.0012
1000,0.0035,0.0006,0.0069,0.0017,0.0006,0.0,0.0006
5000,0.0006,0.0002,0.0003,0.0006,0.0000,0.0,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.0577,0.1155,0.0000,0.0000,0.0000,0.0,0.0000
50,0.0721,0.0416,0.0462,0.0000,0.0000,0.0,0.0115
100,0.0289,0.0300,0.0231,0.0000,0.0000,0.0,0.0000
500,0.0225,0.0070,0.0031,0.0053,0.0000,0.0,0.0031
1000,0.0087,0.0035,0.0086,0.0036,0.0006,0.0,0.0021
5000,0.0005,0.0000,0.0004,0.0012,0.0001,0.0,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
10,0.1732,0.1000,0.2309,0.0000,0.0000,0.0000,0.0000
50,0.0808,0.0115,0.0115,0.0000,0.0000,0.0000,0.0000
100,0.0208,0.0458,0.0265,0.0000,0.0058,0.0000,0.0000
500,0.0058,0.0220,0.0064,0.0012,0.0000,0.0000,0.0012
1000,0.0145,0.0095,0.0031,0.0026,0.0006,0.0006,0.0020
5000,0.0032,0.0004,0.0005,0.0010,0.0001,0.0000,0.0002


In [ ]:
# Calculate mean R@k scores and write to CSV
all_rk_scores_means = calc_score_means(all_rk_scores)
os.makedirs('table-6', exist_ok=True)
pd.DataFrame(all_rk_scores_means[0]).to_csv('table-6/all-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[1]).to_csv('table-6/amino_acid-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[2]).to_csv('table-6/nucleotide-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[3]).to_csv('table-6/multicell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[4]).to_csv('table-6/cell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[5]).to_csv('table-6/other-rk-means.csv', index=False)

# Calculate standard deviations for R@k scores and write to CSV
all_rk_scores_sds = calc_score_sds(all_rk_scores)
os.makedirs('table-13', exist_ok=True)
pd.DataFrame(all_rk_scores_sds[0]).to_csv('table-13/all-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[1]).to_csv('table-13/amino_acid-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[2]).to_csv('table-13/nucleotide-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[3]).to_csv('table-13/multicell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[4]).to_csv('table-13/cell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[5]).to_csv('table-13/other-rk-sds.csv', index=False)

In [ ]:
# Display mean R@k scores console
print("Mean R@k scores:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_means[0].round(4))
    display(all_rk_scores_means[1].round(4))
    display(all_rk_scores_means[2].round(4))
    display(all_rk_scores_means[3].round(4))
    display(all_rk_scores_means[4].round(4))
    display(all_rk_scores_means[5].round(4))

In [ ]:
# Display standard deviation of R@k scores to console
print("R@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_sds[0].round(4))
    display(all_rk_scores_sds[1].round(4))
    display(all_rk_scores_sds[2].round(4))
    display(all_rk_scores_sds[3].round(4))
    display(all_rk_scores_sds[4].round(4))
    display(all_rk_scores_sds[5].round(4))

In [ ]:
# Calculate mean F1@k scores and write to CSV
all_fk_scores_means = calc_score_means(all_fk_scores)
os.makedirs('table-7', exist_ok=True)
pd.DataFrame(all_fk_scores_means[0]).to_csv('table-7/all-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[1]).to_csv('table-7/amino_acid-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[2]).to_csv('table-7/nucleotide-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[3]).to_csv('table-7/multicell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[4]).to_csv('table-7/cell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[5]).to_csv('table-7/other-fk-means.csv', index=False)

# Calculate standard deviations for F1@k scores and write to CSV
all_fk_scores_sds = calc_score_sds(all_fk_scores)
os.makedirs('table-14', exist_ok=True)
pd.DataFrame(all_fk_scores_sds[0]).to_csv('table-14/all-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[1]).to_csv('table-14/amino_acid-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[2]).to_csv('table-14/nucleotide-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[3]).to_csv('table-14/multicell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[4]).to_csv('table-14/cell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[5]).to_csv('table-14/other-fk-sds.csv', index=False)

In [ ]:
# Display mean F1@k scores to console
print("Mean F1@k scores:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_means[0].round(4))
    display(all_fk_scores_means[1].round(4))
    display(all_fk_scores_means[2].round(4))
    display(all_fk_scores_means[3].round(4))
    display(all_fk_scores_means[4].round(4))
    display(all_fk_scores_means[5].round(4))

In [ ]:
# Display standard deviation of F1@k scores to console
print("F1@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_sds[0].round(4))
    display(all_fk_scores_sds[1].round(4))
    display(all_fk_scores_sds[2].round(4))
    display(all_fk_scores_sds[3].round(4))
    display(all_fk_scores_sds[4].round(4))
    display(all_fk_scores_sds[5].round(4))

In [ ]:
# Calculate mean RBO scores
mean_rbo_scores = calc_mean_rbo_scores(all_rbo_scores)

# Write to CSV
os.makedirs('table-8', exist_ok=True)
pd.DataFrame(mean_rbo_scores).to_csv('table-8/rbo-means.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores.round(4))

In [ ]:
# Calculate mean RBO scores by semantic class
mean_rbo_scores2 = calc_mean_rbo_scores(all_rbo_scores2)

# Write to CSV
os.makedirs('table-8', exist_ok=True)
pd.DataFrame(mean_rbo_scores2).to_csv('table-8/rbo-means-by-semantic-class.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores2.round(4))

## Top 10 Ranked Terms Example

Here we reproduce the result of Table 5 from the manuscript.

In [ ]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

In [ ]:
ranked_terms_df = resort(term_scores_df)
top_10_ranked_terms_df = pd.DataFrame(top_k(ranked_terms_df, 10))
display(top_10_ranked_terms_df)

In [ ]:
# Write to CSV
os.makedirs('table-9', exist_ok=True)
top_10_ranked_terms_df.to_csv('table-9/top-10-terms.csv', index=False)

## Stopwords Exploratory Analysis

Here we reproduce the result of Table 6 from the manuscript.

In [ ]:
import pandas as pd
from nltk.corpus import stopwords

# Ensure you have the stopwords downloaded
import nltk
nltk.download('stopwords')

def getrank(sorted_measures):
    unique_terms = set()
    for terms in sorted_measures.values():
        unique_terms.update(terms)
    unique_terms = sorted(unique_terms)
    
    # Create a data frame to hold the rankings
    ranking_df = pd.DataFrame(index=unique_terms, columns=sorted_measures.keys())
    
    # Fill the data frame with rankings
    for measure, terms in sorted_measures.items():
        for rank, term in enumerate(terms):
            ranking_df.at[term, measure] = rank + 1  # Rank starts from 1
    
    # Replace NaN with a large number to indicate unranked terms
    ranking_df = ranking_df.fillna(len(unique_terms) + 1)
    #csv_file_path = 'ranking_table.csv'
    #ranking_df.to_csv(csv_file_path)
    return ranking_df

# Function to filter stopwords from the ranking data frame
def filter_stopwords(ranking_df):
    stopwords_list = set(stopwords.words('english'))
    
    # Filter the data frame to include only stopwords
    stopwords_rank = ranking_df[ranking_df.index.isin(stopwords_list)]
    
    # Save the stopwords ranking data frame to a CSV file
    #csv_file_path = 'stopwords_ranking_table.csv'
    #stopwords_rank.to_csv(csv_file_path)
    
    return stopwords_rank

In [ ]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Generate term dispersion ranks for R different versions of the data
R = 100
all_quantiles_df = []
for r in tqdm(range(R)):
    sorted_measures = resort(term_scores_df)
    rank = getrank(sorted_measures)
    stopwords_ranks_df = filter_stopwords(rank)
    bursty_measure_names = stopwords_ranks_df.head(0)
    quantiles = []
    for bursty_measure_name in bursty_measure_names:
        quantiles.append(stopwords_ranks_df[bursty_measure_name].quantile([0, 0.25, 0.5, 0.75, 1]))
    quantiles_df = pd.DataFrame(quantiles)
    all_quantiles_df.append(quantiles_df)

In [ ]:
# Extract the column and index names from the first quantiles data frame
columns = all_quantiles_df[0].columns
index = all_quantiles_df[0].index

# Initialize empty data frames to store the mean and standard deviation values
mean_df = pd.DataFrame(index=index, columns=columns)
std_df = pd.DataFrame(index=index, columns=columns)

# Compute the mean and standard deviation of corresponding elements across all matrices
for col in columns:
    for idx in index:
        values = [matrix.at[idx, col] for matrix in all_quantiles_df]
        mean_df.at[idx, col] = np.mean(values)
        std_df.at[idx, col] = np.std(values)

In [ ]:
# Display the resulting data frames
print("Mean values:")
with pd.option_context('display.precision', 4):
    display(mean_df)
print("\nStandard deviations:")
with pd.option_context('display.precision', 4):
    display(std_df)

In [ ]:
# Write to CSV
os.makedirs('table-10', exist_ok=True)
mean_df.to_csv('table-10/stopword-rank-means.csv')
std_df.to_csv('table-10/stopword-rank-sds.csv')